# Práctica de repaso de PySpark

Este notebook contiene **10 ejercicios resueltos** para repasar:

- **Joins**
- **Funciones de fecha y tiempo**
- **Funciones matemáticas**
- **Funciones de texto**



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.appName("repaso-pyspark-soluciones").getOrCreate()

clientes_data = [
    (1, "Ana López", "ana.lopez@email.com", "  Madrid  ", "2024-01-15"),
    (2, "Luis García", "luis.garcia@email.com", "Barcelona", "2024-02-10"),
    (3, "Marta Pérez", "marta.perez@email.com", "Valencia ", "2024-03-05"),
    (4, "Juan Ruiz", "juan.ruiz@email.com", "Sevilla", "2024-03-20"),
    (5, "Elena Díaz", "elena.diaz@email.com", "Bilbao", "2024-04-01")
]

clientes_schema = StructType([
    StructField("cliente_id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("email", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("fecha_alta_str", StringType(), True)
])

clientes_df = spark.createDataFrame(clientes_data, schema=clientes_schema)

pedidos_data = [
    (101, 1, "2024-05-01 10:15:30", 120.50, 2, " online "),
    (102, 1, "2024-05-03 18:40:10", 89.99, 1, "tienda"),
    (103, 2, "2024-05-05 09:05:00", 45.00, 3, "online"),
    (104, 3, "2024-05-07 12:00:45", 210.75, 5, " app"),
    (105, 6, "2024-05-08 16:20:00", 15.00, 1, "online"),
    (106, 4, "2024-05-10 21:10:59", 99.95, 2, "tienda ")
]

pedidos_schema = StructType([
    StructField("pedido_id", IntegerType(), False),
    StructField("cliente_id", IntegerType(), True),
    StructField("fecha_pedido_str", StringType(), True),
    StructField("importe", DoubleType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("canal", StringType(), True)
])

pedidos_df = spark.createDataFrame(pedidos_data, schema=pedidos_schema)

pagos_data = [
    (1001, 101, "2024-05-01 10:20:00", "120.50", "tarjeta"),
    (1002, 102, "2024-05-03 18:45:00", "89.99", "paypal"),
    (1003, 103, "2024-05-05 09:10:00", "45.00", "tarjeta"),
    (1004, 104, "2024-05-07 12:10:00", "210.75", "transferencia"),
    (1005, 107, "2024-05-09 08:00:00", "50.00", "tarjeta")
]

pagos_schema = StructType([
    StructField("pago_id", IntegerType(), False),
    StructField("pedido_id", IntegerType(), True),
    StructField("fecha_pago_str", StringType(), True),
    StructField("importe_pagado_str", StringType(), True),
    StructField("metodo_pago", StringType(), True)
])

pagos_df = spark.createDataFrame(pagos_data, schema=pagos_schema)

print("Clientes")
clientes_df.show(truncate=False)

print("Pedidos")
pedidos_df.show(truncate=False)

print("Pagos")
pagos_df.show(truncate=False)


Clientes
+----------+-----------+---------------------+----------+--------------+
|cliente_id|nombre     |email                |ciudad    |fecha_alta_str|
+----------+-----------+---------------------+----------+--------------+
|1         |Ana López  |ana.lopez@email.com  |  Madrid  |2024-01-15    |
|2         |Luis García|luis.garcia@email.com|Barcelona |2024-02-10    |
|3         |Marta Pérez|marta.perez@email.com|Valencia  |2024-03-05    |
|4         |Juan Ruiz  |juan.ruiz@email.com  |Sevilla   |2024-03-20    |
|5         |Elena Díaz |elena.diaz@email.com |Bilbao    |2024-04-01    |
+----------+-----------+---------------------+----------+--------------+

Pedidos
+---------+----------+-------------------+-------+--------+--------+
|pedido_id|cliente_id|fecha_pedido_str   |importe|cantidad|canal   |
+---------+----------+-------------------+-------+--------+--------+
|101      |1         |2024-05-01 10:15:30|120.5  |2       | online |
|102      |1         |2024-05-03 18:40:10|89.99  

## Ejercicio 1. Inner Join

Haz un **inner join** entre `clientes_df` y `pedidos_df` usando `cliente_id`.

Muestra estas columnas:
- `cliente_id`
- `nombre`
- `pedido_id`
- `importe`
- `canal`

Después ordena por `cliente_id` y `pedido_id`.

**Conceptos**: 7.1 Join Types, 7.2 Basic Syntax.

In [ ]:
ej1_df = clientes_df.join(pedidos_df, on="cliente_id", how="inner") \
    .select("cliente_id", "nombre", "pedido_id", "importe", "canal") \
    .orderBy("cliente_id", "pedido_id")

ej1_df.show(truncate=False)

+----------+-----------+---------+-------+--------+
|cliente_id|nombre     |pedido_id|importe|canal   |
+----------+-----------+---------+-------+--------+
|1         |Ana López  |101      |120.5  | online |
|1         |Ana López  |102      |89.99  |tienda  |
|2         |Luis García|103      |45.0   |online  |
|3         |Marta Pérez|104      |210.75 | app    |
|4         |Juan Ruiz  |106      |99.95  |tienda  |
+----------+-----------+---------+-------+--------+



## Ejercicio 2. Left Join y detección de faltantes
Haz un **left join** entre `pedidos_df` y `pagos_df` usando `pedido_id`.

Objetivo:
- mantener todos los pedidos,
- detectar qué pedidos **no tienen pago asociado**.

Muestra:
- `pedido_id`
- `cliente_id`
- `importe`
- `pago_id`
- una nueva columna llamada `tiene_pago` con valores `SI` o `NO`.

**Conceptos**: Join types, `when`, `isNull`.

In [ ]:
ej2_df = pedidos_df.join(pagos_df, on="pedido_id", how="left") \
    .withColumn("tiene_pago", F.when(F.col("pago_id").isNull(), "NO").otherwise("SI")) \
    .select("pedido_id", "cliente_id", "importe", "pago_id", "tiene_pago")

ej2_df.show(truncate=False)


+---------+----------+-------+-------+----------+
|pedido_id|cliente_id|importe|pago_id|tiene_pago|
+---------+----------+-------+-------+----------+
|101      |1         |120.5  |1001   |SI        |
|102      |1         |89.99  |1002   |SI        |
|103      |2         |45.0   |1003   |SI        |
|104      |3         |210.75 |1004   |SI        |
|105      |6         |15.0   |NULL   |NO        |
|106      |4         |99.95  |NULL   |NO        |
+---------+----------+-------+-------+----------+



## Ejercicio 3. Full Join para inconsistencias
Usa un **full join** entre `pedidos_df` y `pagos_df` por `pedido_id`.

Busca:
- pagos sin pedido,
- pedidos sin pago.

Crea una columna `estado_cruce` con alguno de estos valores:
- `OK`
- `PEDIDO_SIN_PAGO`
- `PAGO_SIN_PEDIDO`

**Conceptos**: Full join, validación de datos.


In [ ]:
ej3_df = pedidos_df.join(pagos_df, on="pedido_id", how="full") \
    .withColumn("cliente_id_pedido", F.col("cliente_id")) \
    .withColumn("estado_cruce",
        F.when(F.col("pago_id").isNull(), "PEDIDO_SIN_PAGO")
         .when(F.col("cliente_id").isNull(), "PAGO_SIN_PEDIDO")
         .otherwise("OK")
    ).select("pedido_id", "cliente_id_pedido", "pago_id", "estado_cruce")

ej3_df.show(truncate=False)


+---------+-----------------+-------+---------------+
|pedido_id|cliente_id_pedido|pago_id|estado_cruce   |
+---------+-----------------+-------+---------------+
|101      |1                |1001   |OK             |
|102      |1                |1002   |OK             |
|103      |2                |1003   |OK             |
|104      |3                |1004   |OK             |
|105      |6                |NULL   |PEDIDO_SIN_PAGO|
|106      |4                |NULL   |PEDIDO_SIN_PAGO|
|107      |NULL             |1005   |PAGO_SIN_PEDIDO|
+---------+-----------------+-------+---------------+



## Ejercicio 4. String to Date y Date to String

En `clientes_df`:

1. Convierte `fecha_alta_str` a tipo fecha en una nueva columna `fecha_alta`.
2. Crea otra columna llamada `fecha_alta_formateada` con formato `dd/MM/yyyy`.

**Conceptos**: 8.1 String to Date Format, 8.3 Date to String Format.


In [ ]:
clientes_df = clientes_df \
    .withColumn("fecha_alta", F.to_date(F.col("fecha_alta_str"), "yyyy-MM-dd")) \
    .withColumn("fecha_alta_formateada", F.date_format(F.col("fecha_alta"), "dd/MM/yyyy"))

clientes_df.printSchema()
clientes_df.show(truncate=False)


root
 |-- cliente_id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- email: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- fecha_alta_str: string (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- fecha_alta_formateada: string (nullable = true)

+----------+-----------+---------------------+----------+--------------+----------+---------------------+
|cliente_id|nombre     |email                |ciudad    |fecha_alta_str|fecha_alta|fecha_alta_formateada|
+----------+-----------+---------------------+----------+--------------+----------+---------------------+
|1         |Ana López  |ana.lopez@email.com  |  Madrid  |2024-01-15    |2024-01-15|15/01/2024           |
|2         |Luis García|luis.garcia@email.com|Barcelona |2024-02-10    |2024-02-10|10/02/2024           |
|3         |Marta Pérez|marta.perez@email.com|Valencia  |2024-03-05    |2024-03-05|05/03/2024           |
|4         |Juan Ruiz  |juan.ruiz@email.com  |Sevilla   |2024-

## Ejercicio 5. String to Timestamp, Timestamp to String y funciones de tiempo


En `pedidos_df`:

1. Convierte `fecha_pedido_str` a timestamp en `fecha_pedido_ts`.
2. Crea una columna `hora_pedido` con la hora del pedido.
3. Crea una columna `minuto_pedido` con el minuto.
4. Crea una columna `fecha_pedido_texto` con formato `yyyy/MM/dd HH:mm`.

**Conceptos**: 8.2, 8.4, 8.6.


In [ ]:
pedidos_df = pedidos_df \
    .withColumn("fecha_pedido_ts", F.to_timestamp(F.col("fecha_pedido_str"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("hora_pedido", F.hour(F.col("fecha_pedido_ts"))) \
    .withColumn("minuto_pedido", F.minute(F.col("fecha_pedido_ts"))) \
    .withColumn("fecha_pedido_texto", F.date_format(F.col("fecha_pedido_ts"), "yyyy/MM/dd HH:mm"))

pedidos_df.printSchema()
pedidos_df.show(truncate=False)

root
 |-- pedido_id: integer (nullable = false)
 |-- cliente_id: integer (nullable = true)
 |-- fecha_pedido_str: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- canal: string (nullable = true)
 |-- fecha_pedido_ts: timestamp (nullable = true)
 |-- hora_pedido: integer (nullable = true)
 |-- minuto_pedido: integer (nullable = true)
 |-- fecha_pedido_texto: string (nullable = true)

+---------+----------+-------------------+-------+--------+--------+-------------------+-----------+-------------+------------------+
|pedido_id|cliente_id|fecha_pedido_str   |importe|cantidad|canal   |fecha_pedido_ts    |hora_pedido|minuto_pedido|fecha_pedido_texto|
+---------+----------+-------------------+-------+--------+--------+-------------------+-----------+-------------+------------------+
|101      |1         |2024-05-01 10:15:30|120.5  |2       | online |2024-05-01 10:15:30|10         |15           |2024/05/01 10:15  |
|102      |1     

## Ejercicio 6. Date Functions
Partiendo de `fecha_alta` y `fecha_pedido_ts`:

1. Une clientes y pedidos por `cliente_id`.
2. Calcula cuántos días pasaron entre la fecha de alta del cliente y la fecha del pedido.
3. Extrae además:
   - año del pedido,
   - mes del pedido,
   - día del mes,
   - día de la semana.

**Conceptos**: 8.5 Date Functions.

In [ ]:
ej6_df = clientes_df.join(pedidos_df, on="cliente_id", how="inner") \
    .withColumn("dias_desde_alta", F.datediff(F.col("fecha_pedido_ts"), F.col("fecha_alta"))) \
    .withColumn("anio_pedido", F.year(F.col("fecha_pedido_ts"))) \
    .withColumn("mes_pedido", F.month(F.col("fecha_pedido_ts"))) \
    .withColumn("dia_mes_pedido", F.dayofmonth(F.col("fecha_pedido_ts"))) \
    .withColumn("dia_semana", F.dayofweek(F.col("fecha_pedido_ts"))) \
    .select("cliente_id", "nombre", "pedido_id", "fecha_alta", "fecha_pedido_ts", 
            "dias_desde_alta", "anio_pedido", "mes_pedido", "dia_mes_pedido", "dia_semana")

ej6_df.show(truncate=False)



+----------+-----------+---------+----------+-------------------+---------------+-----------+----------+--------------+----------+
|cliente_id|nombre     |pedido_id|fecha_alta|fecha_pedido_ts    |dias_desde_alta|anio_pedido|mes_pedido|dia_mes_pedido|dia_semana|
+----------+-----------+---------+----------+-------------------+---------------+-----------+----------+--------------+----------+
|1         |Ana López  |101      |2024-01-15|2024-05-01 10:15:30|107            |2024       |5         |1             |4         |
|1         |Ana López  |102      |2024-01-15|2024-05-03 18:40:10|109            |2024       |5         |3             |6         |
|2         |Luis García|103      |2024-02-10|2024-05-05 09:05:00|85             |2024       |5         |5             |1         |
|3         |Marta Pérez|104      |2024-03-05|2024-05-07 12:00:45|63             |2024       |5         |7             |3         |
|4         |Juan Ruiz  |106      |2024-03-20|2024-05-10 21:10:59|51             |20

## Ejercicio 7. Simple Arithmetic
En `pedidos_df`:

1. Calcula el **precio unitario** como `importe / cantidad`.
2. Crea una columna `importe_con_iva` suponiendo un IVA del 21%.
3. Crea una columna `descuento_10` aplicando un 10% de descuento al importe.

**Conceptos**: 9.1 Simple Arithmetic.

In [ ]:
ej7_df = pedidos_df \
    .withColumn("precio_unitario", F.col("importe") / F.col("cantidad")) \
    .withColumn("importe_con_iva", F.col("importe") * 1.21) \
    .withColumn("descuento_10", F.col("importe") * 0.9)

ej7_df.select("pedido_id", "cliente_id", "fecha_pedido_str", "importe", "cantidad", 
              "canal", "precio_unitario", "importe_con_iva", "descuento_10").show(truncate=False)


+---------+----------+-------------------+-------+--------+--------+---------------+------------------+------------+
|pedido_id|cliente_id|fecha_pedido_str   |importe|cantidad|canal   |precio_unitario|importe_con_iva   |descuento_10|
+---------+----------+-------------------+-------+--------+--------+---------------+------------------+------------+
|101      |1         |2024-05-01 10:15:30|120.5  |2       | online |60.25          |145.805           |108.45      |
|102      |1         |2024-05-03 18:40:10|89.99  |1       |tienda  |89.99          |108.88789999999999|80.991      |
|103      |2         |2024-05-05 09:05:00|45.0   |3       |online  |15.0           |54.449999999999996|40.5        |
|104      |3         |2024-05-07 12:00:45|210.75 |5       | app    |42.15          |255.0075          |189.675     |
|105      |6         |2024-05-08 16:20:00|15.0   |1       |online  |15.0           |18.15             |13.5        |
|106      |4         |2024-05-10 21:10:59|99.95  |2       |tiend

## Ejercicio 8. Complex Arithmetic y redondeos
En `pedidos_df`:

1. Crea una columna `importe_redondeado` con 1 decimal.
2. Crea una columna `raiz_importe` usando la raíz cuadrada del importe.
3. Crea una columna `log_importe` usando logaritmo natural.
4. Crea una columna `categoria_importe`:
   - `BAJO` si importe < 50
   - `MEDIO` si importe entre 50 y 100
   - `ALTO` si importe > 100

**Conceptos**: 9.2 Complex Arithmetic.

In [ ]:
ej8_df = pedidos_df \
    .withColumn("importe_redondeado", F.round(F.col("importe"), 1)) \
    .withColumn("raiz_importe", F.sqrt(F.col("importe"))) \
    .withColumn("log_importe", F.log(F.col("importe"))) \
    .withColumn("categoria_importe",
        F.when(F.col("importe") < 50, "BAJO")
         .when((F.col("importe") >= 50) & (F.col("importe") <= 100), "MEDIO")
         .otherwise("ALTO")
    )

ej8_df.select("pedido_id", "cliente_id", "fecha_pedido_str", "importe", "cantidad", 
              "canal", "importe_redondeado", "raiz_importe", "log_importe", "categoria_importe").show(truncate=False)

+---------+----------+-------------------+-------+--------+--------+------------------+------------------+------------------+-----------------+
|pedido_id|cliente_id|fecha_pedido_str   |importe|cantidad|canal   |importe_redondeado|raiz_importe      |log_importe       |categoria_importe|
+---------+----------+-------------------+-------+--------+--------+------------------+------------------+------------------+-----------------+
|101      |1         |2024-05-01 10:15:30|120.5  |2       | online |120.5             |10.977249200050075|4.791649752930709 |ALTO             |
|102      |1         |2024-05-03 18:40:10|89.99  |1       |tienda  |90.0              |9.486305919587455 |4.499698553045857 |MEDIO            |
|103      |2         |2024-05-05 09:05:00|45.0   |3       |online  |45.0              |6.708203932499369 |3.8066624897703196|BAJO             |
|104      |3         |2024-05-07 12:00:45|210.75 |5       | app    |210.8             |14.517231140957975|5.350672596881965 |ALTO       

## Ejercicio 9. Basic String Functions, trim y pad
Trabaja con `clientes_df` y `pedidos_df`.

1. Limpia la columna `ciudad` eliminando espacios sobrantes.
2. Convierte `nombre` a mayúsculas en una nueva columna.
3. Convierte `email` a minúsculas.
4. Limpia la columna `canal` eliminando espacios laterales.
5. Crea una columna `cliente_id_str` rellenando con ceros a la izquierda hasta longitud 4.

**Conceptos**: 10.1 Basic String Functions, 10.2 Trim and Pad Functions.


In [ ]:
clientes_df_str = clientes_df \
    .withColumn("ciudad_limpia", F.trim(F.col("ciudad"))) \
    .withColumn("nombre_mayus", F.upper(F.col("nombre"))) \
    .withColumn("email_minus", F.lower(F.col("email"))) \
    .withColumn("cliente_id_str", F.lpad(F.col("cliente_id").cast("string"), 4, "0"))

pedidos_df_str = pedidos_df \
    .withColumn("canal_limpio", F.trim(F.col("canal")))

clientes_df_str.select("cliente_id", "nombre", "email", "ciudad", "fecha_alta_str", 
                       "ciudad_limpia", "nombre_mayus", "email_minus", "cliente_id_str").show(truncate=False)

pedidos_df_str.select("pedido_id", "cliente_id", "fecha_pedido_str", "importe", 
                      "cantidad", "canal", "canal_limpio").show(truncate=False)

+----------+-----------+---------------------+----------+--------------+-------------+------------+---------------------+--------------+
|cliente_id|nombre     |email                |ciudad    |fecha_alta_str|ciudad_limpia|nombre_mayus|email_minus          |cliente_id_str|
+----------+-----------+---------------------+----------+--------------+-------------+------------+---------------------+--------------+
|1         |Ana López  |ana.lopez@email.com  |  Madrid  |2024-01-15    |Madrid       |ANA LÓPEZ   |ana.lopez@email.com  |0001          |
|2         |Luis García|luis.garcia@email.com|Barcelona |2024-02-10    |Barcelona    |LUIS GARCÍA |luis.garcia@email.com|0002          |
|3         |Marta Pérez|marta.perez@email.com|Valencia  |2024-03-05    |Valencia     |MARTA PÉREZ |marta.perez@email.com|0003          |
|4         |Juan Ruiz  |juan.ruiz@email.com  |Sevilla   |2024-03-20    |Sevilla      |JUAN RUIZ   |juan.ruiz@email.com  |0004          |
|5         |Elena Díaz |elena.diaz@email.

## Ejercicio 10. Advanced String Functions y conversión de tipos
Trabaja con `pagos_df`.

1. Convierte `importe_pagado_str` a tipo numérico (`double`).
2. Extrae el dominio del email de `clientes_df` en una nueva columna `dominio_email`.
3. Crea una tabla final uniendo clientes, pedidos y pagos con estas columnas:
   - `cliente_id`
   - `nombre`
   - `dominio_email`
   - `pedido_id`
   - `metodo_pago`
   - `importe`
   - `importe_pagado`
4. Crea una columna `coincide_importe` que indique si `importe == importe_pagado`.

**Conceptos**: 10.3 Advanced String Functions, 10.4 Converting to Other Data Types.


In [ ]:
pagos_df_casted = pagos_df.withColumn("importe_pagado", F.col("importe_pagado_str").cast("double"))
clientes_df_dom = clientes_df.withColumn("dominio_email", F.split(F.col("email"), "@").getItem(1))

ej10_df = clientes_df_dom.join(pedidos_df, on="cliente_id", how="inner") \
    .join(pagos_df_casted, on="pedido_id", how="left") \
    .withColumn("coincide_importe", F.col("importe") == F.col("importe_pagado")) \
    .select("cliente_id", "nombre", "dominio_email", "pedido_id", "metodo_pago", 
            "importe", "importe_pagado", "coincide_importe")

ej10_df.show(truncate=False)


+----------+-----------+-------------+---------+-------------+-------+--------------+----------------+
|cliente_id|nombre     |dominio_email|pedido_id|metodo_pago  |importe|importe_pagado|coincide_importe|
+----------+-----------+-------------+---------+-------------+-------+--------------+----------------+
|1         |Ana López  |email.com    |101      |tarjeta      |120.5  |120.5         |true            |
|1         |Ana López  |email.com    |102      |paypal       |89.99  |89.99         |true            |
|2         |Luis García|email.com    |103      |tarjeta      |45.0   |45.0          |true            |
|3         |Marta Pérez|email.com    |104      |transferencia|210.75 |210.75        |true            |
|4         |Juan Ruiz  |email.com    |106      |NULL         |99.95  |NULL          |false           |
+----------+-----------+-------------+---------+-------------+-------+--------------+----------------+



## Ejercicio extra opcional
Crea un DataFrame resumen por cliente que incluya:
- número de pedidos,
- importe total gastado,
- fecha del primer pedido,
- fecha del último pedido.

Después formatea la fecha del último pedido como texto con formato `dd-MM-yyyy HH:mm`.


In [ ]:
extra_df = pedidos_df \
    .groupBy("cliente_id") \
    .agg(
        F.count("pedido_id").alias("num_pedidos"),
        F.round(F.sum("importe"), 2).alias("importe_total"),
        F.min("fecha_pedido_ts").alias("primer_pedido"),
        F.max("fecha_pedido_ts").alias("ultimo_pedido")
    ) \
    .withColumn("ultimo_pedido_texto", F.date_format(F.col("ultimo_pedido"), "dd-MM-yyyy HH:mm")) \
    .select("cliente_id", "num_pedidos", "importe_total", "primer_pedido", "ultimo_pedido", "ultimo_pedido_texto")

extra_df.show(truncate=False)



+----------+-----------+-------------+-------------------+-------------------+-------------------+
|cliente_id|num_pedidos|importe_total|primer_pedido      |ultimo_pedido      |ultimo_pedido_texto|
+----------+-----------+-------------+-------------------+-------------------+-------------------+
|1         |2          |210.49       |2024-05-01 10:15:30|2024-05-03 18:40:10|03-05-2024 18:40   |
|2         |1          |45.0         |2024-05-05 09:05:00|2024-05-05 09:05:00|05-05-2024 09:05   |
|3         |1          |210.75       |2024-05-07 12:00:45|2024-05-07 12:00:45|07-05-2024 12:00   |
|4         |1          |99.95        |2024-05-10 21:10:59|2024-05-10 21:10:59|10-05-2024 21:10   |
|6         |1          |15.0         |2024-05-08 16:20:00|2024-05-08 16:20:00|08-05-2024 16:20   |
+----------+-----------+-------------+-------------------+-------------------+-------------------+

